# CNBI posthoc analysis walkthrough
Module-by-module workflow for `project_healthy`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from posthoc_analysis import (
    discover_runs,
    make_epochs_from_gdf,
    posterior_and_lateralized_features,
    cross_validated_side_decoding,
    permutation_test_multiclass_accuracy,
    run_project_pipeline,
)


## 1) Discover runs
Set your project root and inspect tasks.

In [ ]:
project_root = Path('/path/to/Box/CNBI/Attention_distraction/project_healthy')
runs = discover_runs(project_root)
pd.DataFrame([r.__dict__ for r in runs]).head()


## 2) Single-run EEG preprocessing
Creates epochs with shape `(n_trials, 64, n_times)` and labels in `{0,1,2}`.

In [ ]:
example = next(r for r in runs if r.task in {'decoding', 'training'} and r.gdf_path is not None)
epochs, labels, times = make_epochs_from_gdf(example.gdf_path)
epochs.shape, labels.shape, times.shape


## 3) Feature extraction
Posterior ROI + lateralized difference waves in 0.2-0.5s.

In [ ]:
import mne
raw = mne.io.read_raw_gdf(example.gdf_path, preload=False, verbose='ERROR')
X = posterior_and_lateralized_features(epochs, raw.ch_names[:64], times, window=(0.2, 0.5))
X.shape


## 4) Offline decoding stats

In [ ]:
cv = cross_validated_side_decoding(X, labels)
perm = permutation_test_multiclass_accuracy(X, labels, n_perm=100)
cv['accuracy'], perm['p_value']


## 5) Full-project pipeline

In [ ]:
result = run_project_pipeline(project_root, output_dir='outputs', n_perm=200)
result['metrics']
